In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
import shap
import torch
sys.path.insert(0, os.path.abspath('..'))
from scripts.models.nn.classes.factory import build_model
from scripts.models.nn.classes.dataset import load_split
warnings.filterwarnings('ignore')
pplt.rc.update({'figure.dpi':100,'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
PREDSDIR   = CONFIGS['filepaths']['predictions']
FEATDIR    = CONFIGS['filepaths']['features']
WEIGHTDIR  = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
MODELS     = CONFIGS['experiments']
NNCFG      = MODELS['nn']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
SPLIT      = 'valid'
PREDVARS   = ['rh','thetae','thetaestar']
NAMEB      = 'baseline_rh_thetae_thetaestar_lf_lhf_shf'
NAMEK      = 'nonparametric_rh_thetae_thetaestar_lf_lhf_shf'
N_BG       = 50
N_EX       = 50
NSAMP      = 256
VARLABELS  = {
    'rh':r'$RH$',
    'thetae':r'$\theta_e$',
    'thetaestar':r'$\theta_e^*$',
    'lhf':'Latent Heat Flux',
    'shf':'Sensible Heat Flux',
    'lf':'Land Fraction'}

In [ ]:
ds       = xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf')
target   = ds.tp.load()
inputs   = {v:ds[v].load() for v in ['rh','thetae','thetaestar','lhf','shf','lf']}
sp       = ds.ps.load()
landmask = inputs['lf']>0.5

results = []
for name,_ in MODELS['nn']['runs'].items():
    if name.startswith('nonparametric_rh') or name.startswith('baseline_rh'):
        filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
        if os.path.exists(filepath):
            output  = xr.open_dataset(filepath).tp.load()
            aligned = xr.align(target,output,*inputs.values(),join='inner')
            ytrue   = aligned[0]
            ypred   = aligned[1]
            xlist   = dict(zip(inputs.keys(),aligned[2:]))
            results.append((name,ytrue,ypred,xlist))

preds = {}
for name in [NAMEB,NAMEK]:
    fpath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if os.path.exists(fpath):
        preds[name] = xr.open_dataset(fpath).tp.load()

resid_b     = (target-preds[NAMEB]).mean('seed').squeeze()
resid_k     = (target-preds[NAMEK]).mean('seed').squeeze()
delta_resid = resid_k-resid_b

weights  = xr.open_dataset(os.path.join(WEIGHTDIR,f'{NAMEK}_{SPLIT}_weights.nc')).load()
features = xr.open_dataset(os.path.join(FEATDIR,f'{NAMEK}_{SPLIT}_features.nc')).load()
LEVELS   = weights['lev'].values
wmean    = {v:weights['k1'].sel(field=v).mean('seed') for v in PREDVARS}
wstd     = {v:weights['k1'].sel(field=v).std('seed')  for v in PREDVARS}
fmean    = {v:features[v].mean('seed')                for v in PREDVARS}

fieldvars_shap = PREDVARS
localvars_shap = ['lf','lhf','shf']
F,L,Y,dlev,nlevs,_,valid_shap,ref = load_split(
    SPLIT,fieldvars_shap,localvars_shap,SPLITSDIR,targetvar='tp')
nfv,nlv   = len(fieldvars_shap),len(localvars_shap)
X_shap    = np.hstack([F.numpy().reshape(len(F),-1),L.numpy()])
is_land_shap = L[:,0].numpy()>0.5

vcols = {}
for i,v in enumerate(fieldvars_shap):
    vcols[v] = slice(i*nlevs,(i+1)*nlevs)
for j,v in enumerate(localvars_shap):
    vcols[v] = slice(nfv*nlevs+j,nfv*nlevs+j+1)

In [ ]:
def get_r2(ytrue,ypred,dims=None):
    dims  = list(ytrue.dims) if dims is None else dims
    ssres = ((ytrue-ypred)**2).sum(dim=dims,skipna=True)
    sstot = ((ytrue-ytrue.mean(dim=dims,skipna=True))**2).sum(dim=dims,skipna=True)
    return 1-ssres/sstot


def bin_land_ocean(flat_predictor,flat_response,flat_landmask,nbins=10):
    valid      = np.isfinite(flat_predictor)
    binedges   = np.quantile(flat_predictor[valid],np.linspace(0,1,nbins+1))
    bincenters = 0.5*(binedges[:-1]+binedges[1:])
    lmu,lsd,omu,osd = [],[],[],[]
    for i in range(nbins):
        inbin = (flat_predictor>=binedges[i])&(flat_predictor<=binedges[i+1])
        lv = flat_response[inbin& flat_landmask]
        ov = flat_response[inbin&~flat_landmask]
        lv,ov = lv[np.isfinite(lv)],ov[np.isfinite(ov)]
        lmu.append(lv.mean() if len(lv) else np.nan)
        lsd.append(lv.std()  if len(lv) else np.nan)
        omu.append(ov.mean() if len(ov) else np.nan)
        osd.append(ov.std()  if len(ov) else np.nan)
    return bincenters,np.array(lmu),np.array(lsd),np.array(omu),np.array(osd)


def compute_skill_vs_bins(results,varname,landmask,nbins=10):
    _,ytrueb,ypredb,xlistb = results[0]
    _,ytruek,ypredk,xlistk = results[1]
    predictor = xlistb[varname]
    pred2d    = predictor.mean(dim=[d for d in predictor.dims if d not in ('time','lat','lon')])
    r2b       = get_r2(ytrueb,ypredb,dims=['time']).mean('seed').squeeze()
    r2k       = get_r2(ytruek,ypredk,dims=['time']).mean('seed').squeeze()
    delta     = r2k-r2b
    predmean  = pred2d.mean('time') if 'time' in pred2d.dims else pred2d
    return bin_land_ocean(
        predmean.values.ravel(),
        delta.values.ravel(),
        (landmask.values if hasattr(landmask,'values') else landmask).ravel().astype(bool),
        nbins=nbins)


def agg_abs(sv,vcols):
    return {v:np.abs(sv[:,vcols[v]]).sum(axis=1).mean() for v in vcols}


def agg_signed(sv,vcols):
    return {v:sv[:,vcols[v]].sum(axis=1).mean() for v in vcols}

In [ ]:
def plot_r2_maps(results):
    fig,axs = pplt.subplots(nrows=1,ncols=3,refwidth=2,proj='cyl',share=False)
    axs.format(grid=False,coast=True,latlim=LATRANGE,lonlim=LONRANGE,
               collabels=['Baseline','Kernel','Kernel $-$ Baseline'])
    r2b  = get_r2(results[0][1],results[0][2],dims=['time']).mean('seed').squeeze()
    r2k  = get_r2(results[1][1],results[1][2],dims=['time']).mean('seed').squeeze()
    diff = r2k-r2b
    style1 = dict(cmap='Blues',vmin=0,vmax=0.8,levels=17,extend='max')
    style2 = dict(cmap='ColdHot_r',vmin=-0.2,vmax=0.2,levels=9,extend='both')
    axs[0].pcolormesh(r2b.lon,r2b.lat,r2b,**style1)
    im1 = axs[1].pcolormesh(r2k.lon,r2k.lat,r2k,**style1)
    im2 = axs[2].pcolormesh(diff.lon,diff.lat,diff,**style2)
    fig.colorbar(im1,loc='b',cols=(1,2),label='R$^2$',ticks=0.1)
    fig.colorbar(im2,loc='b',cols=3,label='$\Delta$R$^2$',ticks=0.1)
    pplt.show()


def plot_skill_vs_predictors(results,landmask,varnames,nbins=10):
    nv  = len(varnames)
    fig,axs = pplt.subplots(nrows=2,ncols=nv//2,refwidth=2.2,refaspect=1.2,sharex=False,sharey=True)
    axs.format(grid=True,ylabel='$\\Delta R^2$')
    for ax,varname in zip(axs,varnames):
        binc,lmu,lsd,omu,osd = compute_skill_vs_bins(results,varname,landmask,nbins=nbins)
        ax.fill_between(binc,lmu-lsd,lmu+lsd,color='sienna',alpha=0.15)
        ax.fill_between(binc,omu-osd,omu+osd,color='steelblue',alpha=0.15)
        ax.plot(binc,lmu,color='sienna',lw=1.5,label='Land')
        ax.plot(binc,omu,color='steelblue',lw=1.5,label='Ocean')
        ax.axhline(0,color='k',lw=0.8,ls='--')
        ax.format(xlabel=VARLABELS[varname])
    axs[0].legend(loc='lr',ncols=1)
    fig.format(suptitle='Kernel $-$ Baseline $\\Delta R^2$ vs. Predictor Value')
    pplt.show()

In [ ]:
def plot_kernel_weights(wmean,wstd,levels):
    colors = {'rh':'cerulean','thetae':'red','thetaestar':'orange'}
    fig,axs = pplt.subplots(nrows=1,ncols=len(PREDVARS),refwidth=1.6,refaspect=0.5,share=False)
    axs.format(grid=True,ylabel='Pressure (hPa)',xlabel='Weight')
    for ax,varname in zip(axs,PREDVARS):
        ax.plot(wmean[varname].values,levels,color=colors[varname],lw=1.5)
        ax.fill_betweenx(levels,
                         wmean[varname].values-wstd[varname].values,
                         wmean[varname].values+wstd[varname].values,
                         color=colors[varname],alpha=0.2)
        ax.axvline(0,color='k',lw=0.8,ls='--')
        ax.format(title=VARLABELS[varname],yreverse=True)
    fig.format(suptitle='Learned Vertical Kernel Weights')
    pplt.show()


def plot_feature_variance(fmean,landmask):
    fig,axs = pplt.subplots(nrows=1,ncols=len(PREDVARS),refwidth=1.8,share=False)
    for ax,varname in zip(axs,PREDVARS):
        fstd       = fmean[varname].std('time')
        land_vals  = fstd.where( landmask).values.ravel()
        ocean_vals = fstd.where(~landmask).values.ravel()
        land_vals  = land_vals [np.isfinite(land_vals )]
        ocean_vals = ocean_vals[np.isfinite(ocean_vals)]
        bins = np.linspace(np.nanmin(fstd.values),np.nanmax(fstd.values),40)
        ax.hist(land_vals, bins=bins,color='sienna',   alpha=0.6,label='Land', density=True)
        ax.hist(ocean_vals,bins=bins,color='steelblue',alpha=0.6,label='Ocean',density=True)
        ax.format(title=VARLABELS[varname],xlabel='Temporal std of feature',ylabel='Density',grid=True)
        ax.legend(loc='upper right',ncols=1)
    fig.format(suptitle='Kernel Feature Variance: Land vs. Ocean')
    pplt.show()


def plot_residual_vs_sp(delta_resid,sp,landmask,nbins=10):
    spmean  = sp.mean('time')
    dr_mean = delta_resid.mean('time')
    bincenters,lmu,lsd,omu,osd = bin_land_ocean(
        spmean.values.ravel(),
        dr_mean.values.ravel(),
        landmask.values.ravel().astype(bool),
        nbins=nbins)
    fig,ax = pplt.subplots(refwidth=3.5)
    ax.fill_between(bincenters,lmu-lsd,lmu+lsd,color='sienna',   alpha=0.15)
    ax.fill_between(bincenters,omu-osd,omu+osd,color='steelblue',alpha=0.15)
    ax.plot(bincenters,lmu,color='sienna',   lw=1.5,label='Land')
    ax.plot(bincenters,omu,color='steelblue',lw=1.5,label='Ocean')
    ax.axhline(0,color='k',lw=0.8,ls='--')
    ax.format(xlabel='Surface pressure (hPa)',ylabel='Mean $\\Delta$ residual (Kernel $-$ Baseline)',
              title='Delta Residual vs. Surface Pressure',grid=True)
    ax.legend(loc='best',ncols=1)
    pplt.show()

In [ ]:
def plot_shap_attribution(sv_bl,sv_bo,sv_kl,sv_ko,vcols,allvars):
    labels = [VARLABELS.get(v,v) for v in allvars]
    xp     = np.arange(len(allvars))
    w      = 0.18
    diff_land  = {v:agg_signed(sv_kl,vcols)[v]-agg_signed(sv_bl,vcols)[v] for v in vcols}
    diff_ocean = {v:agg_signed(sv_ko,vcols)[v]-agg_signed(sv_bo,vcols)[v] for v in vcols}

    fig,axs = pplt.subplots(ncols=2,refwidth=3.2,share=False)

    for i,(mlabel,svl,svo) in enumerate([('Baseline',sv_bl,sv_bo),
                                          ('Kernel',  sv_kl,sv_ko)]):
        for j,(stype,sv,c) in enumerate([('Land',svl,'sienna'),
                                          ('Ocean',svo,'steelblue')]):
            vals = [agg_abs(sv,vcols)[v] for v in allvars]
            axs[0].bar(xp+(i*2+j-1.5)*w,vals,width=w,color=c,
                       alpha=0.5+0.25*i,label=f'{mlabel} {stype}')

    axs[0].set_xticks(xp)
    axs[0].set_xticklabels(labels,rotation=30,ha='right')
    axs[0].format(ylabel='Mean |SHAP|',
                  title='(a) Feature Importance by Model & Surface',grid=True)
    axs[0].legend(loc='ur',ncols=1,fontsize=7)

    axs[1].bar(xp-0.15,[diff_land[v]  for v in allvars],width=0.28,
               color='sienna',   label='Land')
    axs[1].bar(xp+0.15,[diff_ocean[v] for v in allvars],width=0.28,
               color='steelblue',label='Ocean')
    axs[1].axhline(0,color='k',lw=0.8,ls='--')
    axs[1].set_xticks(xp)
    axs[1].set_xticklabels(labels,rotation=30,ha='right')
    axs[1].format(ylabel=r'Mean $\overline{\Delta\phi}$ (Kernel $-$ Baseline)',
                  title='(b) SHAP Difference Attribution (Eq. B.3)',grid=True)
    axs[1].legend(loc='lr',ncols=1,fontsize=7)

    fig.format(suptitle='Kernel SHAP: Error Attribution for Kernel vs. Baseline')
    pplt.show()


def plot_mplots(results,landmask,nbins=15):
    _,yt_b,yp_b,xl_b = results[0]
    _,yt_k,yp_k,xl_k = results[1]

    obs  = yt_b.squeeze()
    base = yp_b.mean('seed').squeeze() if 'seed' in yp_b.dims else yp_b.squeeze()
    kern = yp_k.mean('seed').squeeze() if 'seed' in yp_k.dims else yp_k.squeeze()

    pvars = ['rh','thetae','thetaestar','lhf','shf']
    nv    = len(pvars)

    lm_flat = np.broadcast_to(landmask.values,obs.shape).ravel().astype(bool)

    fig,axs = pplt.subplots(nrows=2,ncols=nv,refwidth=2,refaspect=1.2,
                             sharex=False,sharey='row')
    axs.format(grid=True)

    for col,vn in enumerate(pvars):
        feat = xl_b[vn]
        extra = [d for d in feat.dims if d not in ('time','lat','lon')]
        if extra:
            feat = feat.mean(dim=extra)

        ff = feat.values.ravel()
        fo = obs.values.ravel()
        fb = base.values.ravel()
        fk = kern.values.ravel()

        ok    = np.isfinite(ff) & np.isfinite(fo)
        edges = np.quantile(ff[ok],np.linspace(0,1,nbins+1))
        ctrs  = 0.5*(edges[:-1]+edges[1:])

        for row,(stype,smask) in enumerate([
            ('Land', lm_flat),
            ('Ocean',~lm_flat)]):

            ax = axs[row,col]
            m_obs,m_base,m_kern = [],[],[]
            for i in range(nbins):
                sel = ok & smask & (ff>=edges[i]) & (ff<=edges[i+1])
                if sel.sum()>10:
                    m_obs.append(fo[sel].mean())
                    m_base.append(fb[sel].mean())
                    m_kern.append(fk[sel].mean())
                else:
                    m_obs.append(np.nan)
                    m_base.append(np.nan)
                    m_kern.append(np.nan)

            ax.plot(ctrs,m_obs, 'k-', lw=1.5,label='Obs')
            ax.plot(ctrs,m_base,'-',  lw=1.5,
                    color='sienna' if stype=='Land' else 'steelblue',
                    label='Baseline')
            ax.plot(ctrs,m_kern,'--', lw=1.5,
                    color='darkred' if stype=='Land' else 'navy',
                    label='Kernel')
            ax.format(xlabel=VARLABELS.get(vn,vn))
            if col==0:
                ax.format(ylabel=f'{stype}\nMean Precip')
            if row==0 and col==0:
                ax.legend(loc='ul',ncols=1,fontsize=7)

    fig.format(suptitle='M-Plots: Conditional Average Precipitation by Feature\n'
               '(Where do the models deviate from ground truth?)')
    pplt.show()

In [ ]:
plot_r2_maps(results)

In [ ]:
plot_skill_vs_predictors(results,landmask,varnames=VARLABELS,nbins=10)

In [ ]:
plot_kernel_weights(wmean,wstd,LEVELS)

In [ ]:
plot_feature_variance(fmean,landmask)

In [ ]:
plot_residual_vs_sp(delta_resid,sp,landmask)

## SHAP-Based Error Attribution & M-Plots

Following the framework from [Grundner et al. (2024, JAMES)](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2023MS003763) Appendix B.1, we use **kernel SHAP** to decompose the prediction difference between the kernel and baseline models into per-feature attributions. The key identity (Eq. B.3) is:

$$\bar{f} - \bar{g} \approx \sum_{i \in I \cap J} \overline{(\phi_{f,X,i} - \phi_{g,X,i})}$$

We compute SHAP values on sub-sampled **land** and **ocean** data separately, then compare the mean absolute feature attributions to identify which inputs drive the kernel model's disproportionate skill loss over land.

**M-plots** (conditional average plots) complement this by plotting ground-truth precipitation alongside both models' predictions as a function of each feature, revealing precisely where each model deviates from observations.

In [ ]:
rng = np.random.default_rng(42)

def _load_model(name):
    rc  = NNCFG['runs'][name]
    mdl = build_model(name,rc,nlevs)
    mdl.load_state_dict(torch.load(
        os.path.join(MODELSDIR,'nn',f'{name}_42.pth'),
        map_location='cpu',weights_only=True))
    mdl.eval()
    return mdl,rc['kind'] not in ('baseline','hurdle')

model_base,hk_base = _load_model(NAMEB)
model_kern,hk_kern = _load_model(NAMEK)

def _make_predict_fn(mdl,haskernel):
    def predict(Xin):
        Xin = np.atleast_2d(Xin).astype(np.float32)
        n   = Xin.shape[0]
        ft  = torch.from_numpy(Xin[:,:nfv*nlevs].reshape(n,nfv,nlevs))
        lt  = torch.from_numpy(Xin[:,nfv*nlevs:])
        with torch.no_grad():
            if haskernel:
                out = mdl(ft,dlev,lt,mask=None)
            else:
                out = mdl(ft,lt,mask=torch.ones(n,nlevs))
        return np.atleast_1d(out.numpy())
    return predict

fn_base = _make_predict_fn(model_base,hk_base)
fn_kern = _make_predict_fn(model_kern,hk_kern)

land_idx   = np.where(is_land_shap)[0]
ocean_idx  = np.where(~is_land_shap)[0]
bg_data    = X_shap[rng.choice(len(X_shap),N_BG,replace=False)]
land_samp  = rng.choice(land_idx,min(N_EX,len(land_idx)),replace=False)
ocean_samp = rng.choice(ocean_idx,min(N_EX,len(ocean_idx)),replace=False)

print(f'Kernel SHAP: {X_shap.shape[1]} features, '
      f'{len(land_samp)} land + {len(ocean_samp)} ocean samples, {NSAMP} coalitions')

explainer_b = shap.KernelExplainer(fn_base,bg_data)
explainer_k = shap.KernelExplainer(fn_kern,bg_data)

print('  Baseline - land ...')
sv_bl = explainer_b.shap_values(X_shap[land_samp], nsamples=NSAMP,silent=True)
print('  Baseline - ocean ...')
sv_bo = explainer_b.shap_values(X_shap[ocean_samp],nsamples=NSAMP,silent=True)
print('  Kernel - land ...')
sv_kl = explainer_k.shap_values(X_shap[land_samp], nsamples=NSAMP,silent=True)
print('  Kernel - ocean ...')
sv_ko = explainer_k.shap_values(X_shap[ocean_samp],nsamples=NSAMP,silent=True)
print('Done.')

In [ ]:
plot_shap_attribution(sv_bl,sv_bo,sv_kl,sv_ko,vcols,list(vcols.keys()))

In [ ]:
plot_mplots(results,landmask)